# Exploration for Q6: a measure of need that is not demand
Owner: Valentino. Read `questions/q6_divergence_measure/README.md` and `addendum.md` first.

Order of work, and the rule: **nothing in step 0 is written after a result is seen.**

0. Definition of need, inputs with their defence, rejection and revision criteria
1. What Q1 already established, and the Wilkin test case
2. Build the index (not before step 0 is filled in)
3. Divergence against each of the three metrics
4. Stability under other weights
5. Map, and county characteristics that explain the divergence

## Step 0: written before any data

**Definition of need (one sentence that could turn out to be false):**

> A household is in need if, at any point in the past year, it worried about running out of
> food, regardless of whether it sought help.

This is the first item of the USDA food-security questionnaire, so a survey can measure it
directly and disagree with us. "Regardless of whether it sought help" is what separates
need from demand: the worry exists whether or not the household asks anyone. A county's
need is the number (or share) of its households in that state.

**Inputs, argued one by one.** An input belongs only if all three answers hold up, and
the mechanism has to end in "...so the household is more likely to worry about running
out of food". If it ends in "...so the household is less likely to visit a food shelf",
it is a barrier, not need, and it goes to step 5.

| input | measurable at zero demand? | in MMG? | mechanism | have it? |
|---|---|---|---|---|
| `pct_hh_snap` | yes | no | a household on SNAP has already had to ask for help with food; benefits run out before the month does, so it worries about running out of food | yes (ACS B22003) |
| `pct_no_vehicle` | yes | no | without a vehicle there is no way to carry a week's groceries home, so the household worries about running out of food. (It is also a barrier to reaching a food shelf; it is used on the need side only, never in step 5) | yes (ACS B08201) |
| `senior_poverty_rate` | yes | no (MMG uses all-ages poverty) | poor seniors live on a fixed income with no savings and no future earnings, so a bill they cannot defer becomes a month they worry about food | yes (ACS B17001) |
| housing-cost burden | yes | no | rent or mortgage is paid first; a household paying over 30% of income on housing has little left, so any other bill means worrying about food | no: ACS B25070 (renters) and B25091 (owners), one more pull |
| low access to a grocery store (LILA share) | yes | no | low income and no store within reach: the household worries first about how to get food at all, then how to afford it | partly: USDA Atlas is in the repo (Q8), needs a county aggregate |
| educational attainment (share of adults without a high-school diploma) | yes | no | low education closes doors when trouble comes, so a household that gets into trouble has fewer ways out and worries about food | no: ACS B15003, one more pull |
| credit delinquency | yes | no | a household unable to pay back what it borrowed has to keep finding new financing, so it worries about affording food. Doubt: delinquents were once approved for credit, so they had some standing; the poorest households often have no credit file at all | no: Urban Institute county data, new source. Last priority, because of the doubt |

**Barriers, kept for step 5 (why need does not become a visit):** `mean_km_to_site`,
`rucc_2023`, `pct_senior` (mobility, reluctance to ask). Not inputs to need.

**Left out:** `poverty_rate`, `median_hh_income`, `unemployment_rate` (MMG inputs);
everything derived from activity (demand).

**Evidence that would make us reject the definition:**

> Survey the counties we rank highest on the measure. If the share of households that
> worried about running out of food is no higher than in the rest of the state, the
> measure is wrong.

**Evidence that would make us revise the measure (definition survives, implementation does not):**

> 1. The index is poverty in a costume: regressed on the poverty rate it leaves nothing
>    behind (R2 near 0.9 or above, or the top ten counties identical to the poverty top ten).
>    Need should correlate with poverty; it should not *be* poverty.
> 2. An input's mechanism fails on inspection, or the input turns out to track activity
>    (downstream of visits). Drop it and rebuild.
> 3. The top ten changes under reasonable alternative weights. Then the ranking is not a
>    finding.

**Prediction for Wilkin:** the measure must give Wilkin a need estimate although it has
no recorded activity. The definition allows it: Wilkin has need as long as people live
there.

> Written 1 October, before the index exists: Wilkin ranks around the 70th percentile of
> need (roughly 25th of 87 from the top), driven by SNAP households and no-vehicle
> households, both above the state median. On MMG's own inputs it is below median
> (poverty rate rank 68), so the measure and MMG should disagree on this county.

## Step 1: what Q1 established
- Recorded activity rises faster than need (log-log slope 1.15 to 1.24).
- The Q1 residual is demand relative to MMG need: positive = more recorded than need predicts.
- The residual partly measures the network: counties with more sites record more activity
  per food-insecure person. Pounds is a supply measure as much as a demand measure.
- Distance, RUCC group, no-vehicle households and SNAP households explain 3% of the residual.
- Moran's I = 0.06: no general spatial pattern; border pairs are examples, not evidence.
- Wilkin has food-insecure people and no reporting food shelf.

So the Q1 residual is not a need measure, and the Q6 measure must not be built from it.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "qkit.py").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

import config as cfg
import qkit

In [ ]:
# County-year frame, as in Q1: activity + MMG need + public covariates, 2022-2024
df = qkit.county_year(cfg.Q1_YEARS).merge(
    qkit.covariates(cfg.Q1_YEARS).drop(columns="county_name"),
    on=["fips", "year"], how="left", validate="1:1")
no_activity = df["households"].isna() & df["fi_persons"].notna()
for m in cfg.THREE_METRICS:
    df.loc[no_activity, m] = 0
df.loc[no_activity, "months_observed"] = 0

# The Q1 residual ranking, from the script that rebuilds it
q1 = pd.read_csv(cfg.TAB / "q1_residual_ranking.csv", dtype={"fips": str})

print(df.shape, "|", df["fips"].nunique(), "counties |", sorted(df["year"].unique()))
print("columns available as inputs:")
print(sorted(c for c in df.columns if c not in cfg.THREE_METRICS))

In [ ]:
# The test case: what do we know about Wilkin without any activity data?
cols = ["year", "fi_persons", "fi_rate", "population", "poverty_rate", "pct_no_vehicle",
        "pct_hh_snap", "pct_senior", "rucc_2023", "n_sites_reporting", "mean_km_to_site"]
df[df["county_name"] == "Wilkin"][cols]

## Step 2: the index
Decisions, recorded before any ranking was seen (1 October):

| decision | choice | why |
|---|---|---|
| inputs | the seven in step 0: SNAP, no vehicle, senior poverty, housing burden, no HS diploma, LILA share, debt in collections | each argued above |
| LILA definition | `lila_share_1and10`, USDA's headline | easiest to defend; the vehicle version double-counts `pct_no_vehicle` and is kept for the stability check |
| scale | z-score of each input within each year | puts shares of different sizes on one footing; a coefficient of 1 = one standard deviation |
| weights | equal, 1/7 each | no argument was found for more; alternatives come in step 4 as a stability check, not before |
| direction | higher = more need, for every input | |
| county score | mean of the three years | the ranking is a 2022-2024 statement, as in Q1 |

The index is relative: it ranks counties, it does not say what share of households worried.
That share is what the definition is about and only a survey measures it, which is why the
rejection criterion is a survey.

**Checks that follow from the revision criteria** (results in the next cell):
1. Is it poverty in a costume? Regress the index on the poverty rate; R2 near 0.9 or above
   rejects it. Compare the top ten with the poverty top ten.
2. Does one input dominate? Correlations among inputs, and the first principal component's
   share and loadings. One input carrying most of the variance means the index is that input.
3. Wilkin against the prediction in step 0.

In [ ]:
INPUTS = ["pct_hh_snap", "pct_no_vehicle", "senior_poverty_rate", "pct_hh_housing_burden",
          "pct_no_hs_diploma", "lila_share_1and10", "pct_debt_collections"]
WEIGHTS = pd.Series(1 / len(INPUTS), index=INPUTS)

def build_index(frame, inputs=INPUTS, weights=None):
    """Z-score each input within year, weighted mean -> need index per county-year.
    Weights default to equal over whatever inputs are given (a fixed Series indexed by
    INPUTS silently dropped any swapped-in input, which broke the LILA-vehicle variant)."""
    w = pd.Series(1.0, index=inputs) if weights is None else pd.Series(weights, index=inputs)
    z = frame.groupby("year")[inputs].transform(lambda s: (s - s.mean()) / s.std())
    return (z * w).sum(axis=1) / w.sum()

df["need_index"] = build_index(df)
need = (df.groupby(["fips", "county_name"], as_index=False)
          .agg(need_index=("need_index", "mean"), poverty_rate=("poverty_rate", "mean"),
               fi_rate=("fi_rate", "mean"), population=("population", "mean")))
need["need_rank"] = need["need_index"].rank(ascending=False).astype(int)
need = need.sort_values("need_rank").reset_index(drop=True)

w = need[need["county_name"] == "Wilkin"].iloc[0]
print(f"Wilkin: index {w['need_index']:.2f}, rank {w['need_rank']} of 87 "
      f"(percentile {100 * (1 - (w['need_rank'] - 0.5) / 87):.0f}). Prediction: ~70th percentile, ~25th.")
print("top 10 need:   ", ", ".join(need["county_name"].head(10)))
print("bottom 10 need:", ", ".join(need["county_name"].tail(10)[::-1]))
need[["need_rank", "county_name", "need_index", "poverty_rate", "fi_rate"]].head(15).round(2)

In [ ]:
# Check 1: poverty in a costume?
fit = smf.ols("need_index ~ poverty_rate", data=need).fit()
pov_top = set(need.nlargest(10, "poverty_rate")["county_name"])
mmg_top = set(need.nlargest(10, "fi_rate")["county_name"])
top = set(need.head(10)["county_name"])
print(f"index on poverty rate: R2 = {fit.rsquared:.2f} (revise if ~0.9 or above)")
print(f"top ten shared with poverty top ten: {len(top & pov_top)} | with MMG fi_rate top ten: {len(top & mmg_top)}")
print("Spearman with poverty rate:", round(need["need_index"].corr(need["poverty_rate"], method="spearman"), 2),
      "| with MMG fi_rate:", round(need["need_index"].corr(need["fi_rate"], method="spearman"), 2))

# Check 2: does one input dominate?
z = df.groupby("year")[INPUTS].transform(lambda s: (s - s.mean()) / s.std())
print("\ncorrelations among inputs (county-years):")
display(z.corr().round(2))
U, S, Vt = np.linalg.svd(z.values - z.values.mean(axis=0), full_matrices=False)
print("share of input variance on the first component:", round(S[0] ** 2 / (S ** 2).sum(), 2))
print("loadings on it:", dict(zip(INPUTS, Vt[0].round(2))))
print("correlation of each input with the index:")
print(z.corrwith(df["need_index"]).round(2).to_string())

## Step 3: where need and recorded activity diverge
Divergence is computed for each of the three metrics, as the brief asks. Two views:

1. **Rate against rate.** Activity per 1,000 residents (mean of 2022-2024), ranked, against
   the need rank. `divergence = activity_rank - need_rank`, so **positive = the county
   ranks higher on need than on recorded activity**: need that is not showing up as demand.
   Wilkin has zero activity and ranks last on every metric.
2. **Against Q1's residual.** The Q1 residual is activity relative to MMG need. If the new
   measure captures need MMG misses, counties high on the index should sit below the Q1
   line. A county high on need and negative on the residual is divergent on both views.

**Prediction, written before running (1 October):** Wilkin, Nobles and Grant are in the
top of the divergence list. Nobles and Grant were among the Q1 counties recording less
than need predicts, and Nobles is 3rd on the index.

In [ ]:
act = (df.groupby(["fips", "county_name"])
         .apply(lambda g: pd.Series({m + "_per_1k": 1000 * g[m].sum() / g["population"].sum()
                                     for m in cfg.THREE_METRICS}), include_groups=False)
         .reset_index())
div = need.merge(act, on=["fips", "county_name"])
for m in cfg.THREE_METRICS:
    div[m + "_rank"] = div[m + "_per_1k"].rank(ascending=False).astype(int)
    div["div_" + m] = div[m + "_rank"] - div["need_rank"]
div["div_mean"] = div[["div_" + m for m in cfg.THREE_METRICS]].mean(axis=1)
div = div.sort_values("div_mean", ascending=False).reset_index(drop=True)

print("Spearman, need index vs activity per 1,000 residents:",
      {m: round(div["need_index"].corr(div[m + "_per_1k"], method="spearman"), 2) for m in cfg.THREE_METRICS})
pred = ["Wilkin", "Nobles", "Grant"]
print("predicted counties, position in the divergence list:",
      {c: int(div.index[div["county_name"] == c][0]) + 1 for c in pred})
print("\nneed ranks higher than recorded activity (top 12):")
cols = ["county_name", "need_rank"] + [m + "_rank" for m in cfg.THREE_METRICS] + ["div_mean"]
display(div[cols].head(12).round(1))
print("recorded activity ranks higher than need (top 8):")
div[cols].tail(8)[::-1].round(1)

In [ ]:
# View 1 as a chart: need rank against activity rank, one panel per metric.
# On the diagonal, need and recorded activity agree. Below it: need outranks activity.
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), sharey=True)
for ax, m in zip(axes, cfg.THREE_METRICS):
    ax.scatter(div["need_rank"], div[m + "_rank"], alpha=0.6)
    ax.plot([1, 87], [1, 87], color="grey", ls="--", lw=0.8, label="need and activity agree")
    d = div[m + "_rank"] - div["need_rank"]
    for idx in pd.concat([d.nlargest(6), d.nsmallest(4)]).index:
        ax.annotate(div.loc[idx, "county_name"], (div.loc[idx, "need_rank"], div.loc[idx, m + "_rank"]),
                    xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.invert_xaxis(); ax.invert_yaxis()          # rank 1 at the top right, like a league table
    ax.set_xlabel("need rank (1 = most need)")
    ax.set_title(f"{m} per 1,000 residents")
axes[0].set_ylabel("activity rank (1 = most recorded)")
axes[0].legend(fontsize=8, loc="lower left")
fig.suptitle("Where need outranks recorded activity (below the line) and the reverse (above)")
plt.tight_layout()
plt.show()

In [ ]:
# View 2: the index against Q1's residual (activity relative to MMG need)
both = need.merge(q1[["fips", "mean_resid"]], on="fips", how="left")
print("Spearman, need index vs Q1 residual:", round(both["need_index"].corr(both["mean_resid"], method="spearman"), 2),
      "(Wilkin has no residual)")
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(both["need_index"], both["mean_resid"], c=both["need_rank"], cmap="viridis_r",
                vmin=1, vmax=87, alpha=0.85, edgecolor="white", lw=0.5)
cb = plt.colorbar(sc, ax=ax); cb.set_label("need rank (1 = most need)"); cb.ax.invert_yaxis()
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(0, color="grey", lw=0.8)
both = both.merge(div[["fips", "div_mean"]], on="fips")
show = pd.concat([both.nlargest(8, "div_mean"), both.nsmallest(4, "div_mean"), both.nlargest(3, "need_index")])
for _, r in show.drop_duplicates("fips").iterrows():
    ax.annotate(r["county_name"], (r["need_index"], r["mean_resid"]), xytext=(4, 4),
                textcoords="offset points", fontsize=8)
ax.set_xlabel("need index (higher = more need)")
ax.set_ylabel("Q1 residual: log activity relative to MMG need")
ax.set_title("View 2: the index against what Q1 left unexplained. Labels: most divergent counties")
plt.tight_layout()
plt.show()
print("high need, below the Q1 line (need rank <= 20 and residual < 0):")
both[(both["need_rank"] <= 20) & (both["mean_resid"] < 0)].sort_values("need_rank")[["need_rank", "county_name", "need_index", "mean_resid"]].round(2)

## Step 4: stability
The README's rule: show stability or do not show a ranking. Each variant is compared with
the baseline on three things: how many of the baseline top ten it keeps, the rank
correlation of the full ordering, and where Wilkin lands.

| variant | what it tests |
|---|---|
| drop SNAP and debt | the two inputs closest to poverty (step 2 check) |
| LILA vehicle definition | the other USDA definition |
| rank scaling | whether z-scores let a few extreme counties drive the index |
| leave one input out, seven times | whether any single input carries the ranking |
| 500 random weight draws | how often each county is in the top ten under any weights |

**Prediction, written before running (2 October):** dropping SNAP and debt moves the top ten most. They are the two inputs closest to the index (0.79 each) and the two closest to poverty.

**Result (2 October):** the prediction held; dropping SNAP and debt was the variant that moved the top ten most, and it still kept 7 of 10. The ranking is stable to the tests: the top seven (Mahnomen, Clearwater, Beltrami, Wadena, Ramsey, Pine, Nobles) held under every variant and in at least 71% of the 500 random weightings, while places 8 to 10 rotated among seven counties (the LILA-vehicle variant, once a weighting bug was fixed, brings in Benton and Chippewa), which is light reshuffling at the tail. Wilkin usually places around 27th, close to the equal-weight model, but with a wide range (2nd to 77th): the random draws include extreme weights, and Wilkin is above average on some inputs and below on others, so it moves with the weights and settles near the prediction only on average. That is an argument for keeping equal weights rather than tuning them.

In [ ]:
def rank_index(frame, inputs):
    r = frame.groupby("year")[inputs].rank(pct=True)
    return r.mean(axis=1)

def county_scores(score):
    s = df.assign(score=score).groupby("county_name")["score"].mean()
    return s.rank(ascending=False).astype(int)

base = county_scores(df["need_index"])
base_top = set(base[base <= 10].index)

variants = {
    "drop SNAP and debt": county_scores(build_index(df, [i for i in INPUTS if i not in ("pct_hh_snap", "pct_debt_collections")])),
    "LILA vehicle": county_scores(build_index(df, [i if i != "lila_share_1and10" else "lila_share_vehicle" for i in INPUTS])),
    "rank scaling": county_scores(rank_index(df, INPUTS)),
}
for i in INPUTS:
    variants["without " + i] = county_scores(build_index(df, [j for j in INPUTS if j != i]))

rows = []
for name, r in variants.items():
    rows.append({"variant": name, "top10 kept": len(base_top & set(r[r <= 10].index)),
                 "spearman with baseline": round(base.corr(r, method="spearman"), 3),
                 "Wilkin rank": int(r["Wilkin"]),
                 "enters top 10": ", ".join(sorted(set(r[r <= 10].index) - base_top))})
stab = pd.DataFrame(rows)
print("baseline top ten:", ", ".join(base[base <= 10].sort_values().index))
display(stab)

# Random weights: 500 draws from a flat Dirichlet, i.e. any weights that sum to one
rng = np.random.default_rng(0)
hits = pd.Series(0, index=base.index)
wilkin = []
for _ in range(500):
    r = county_scores(build_index(df, INPUTS, rng.dirichlet(np.ones(len(INPUTS)))))
    hits[r[r <= 10].index] += 1
    wilkin.append(int(r["Wilkin"]))
share = (hits / 500).sort_values(ascending=False)
print("\nshare of 500 random weightings in which each county is in the top ten:")
print(share.head(15).round(2).to_string())
print(f"Wilkin rank across draws: median {int(np.median(wilkin))}, range {min(wilkin)} to {max(wilkin)}")

## Step 5: what explains the divergence, and the map
Same design as Q1's sub-question 4: county-years, year dummies, errors clustered on county.
Outcome: `divergence_z`, the need index minus the mean z-score of activity per 1,000
residents across the three metrics, within year. Positive = need outranks recorded activity.
The explanatory variables are the barriers set aside in step 0, plus the supply side, which
the README puts here and not in the index.

| variable | mechanism | predicted sign |
|---|---|---|
| `mean_km_to_site` | time cost of reaching a site | + |
| `nonmetro_town`, `rural` (vs metro) | further from everything | + |
| `pct_senior` | mobility, reluctance to ask | + |
| `sites_per_10k` | more sites, more gets recorded | − |

**Predictions written before running (2 October), as above.** Sites and distance measure
the same thing from two sides (more sites per resident, shorter distance), so the model is
fitted twice: barriers alone, then with sites added. In Q1 adding sites weakened distance;
expect the same here.

Note what this model cannot do: `pct_no_vehicle` is inside the index, so it cannot also
explain the divergence (step 0).

**Result (2 October):** three of four signs as predicted; seniors came out negative, with an interval too wide to say anything. Distance remains the single variable most likely to have an effect: it does not clear 0.05 (p = 0.075) but has the narrowest interval of the four (-0.02 to 0.44), which suggests that longer distances inhibit need from becoming recorded activity. The RUCC groups have large coefficients (about +0.5) with intervals from slightly negative to very positive: big and uncertain. Adding sites moved distance from 0.21 to 0.18, the Q1 pattern again, milder. The barriers explain 9% of the divergence, against 3% of Q1's residual: the barriers named in step 0 explain more of the gap between this measure and activity than of the gap between MMG and activity, which is consistent with the measure capturing need that barriers keep from becoming demand. Same power limit as Q1: 87 counties see only large effects.

In [ ]:
z_act = (df.assign(**{m + "_per_1k": 1000 * df[m] / df["population"] for m in cfg.THREE_METRICS})
           .groupby("year")[[m + "_per_1k" for m in cfg.THREE_METRICS]]
           .transform(lambda s: (s - s.mean()) / s.std()).mean(axis=1))
q5 = df[["fips", "county_name", "year", "need_index", "mean_km_to_site", "rucc_2023",
         "pct_senior", "sites_per_10k"]].copy()
q5["divergence_z"] = df["need_index"] - z_act
# z-scores within year: the outcome is within-year, so a covariate with a time trend
# (internet access rises every year) must not be allowed to load onto the year dummies
for v in ["mean_km_to_site", "pct_senior", "sites_per_10k"]:
    q5[v + "_z"] = q5.groupby("year")[v].transform(lambda s: (s - s.mean()) / s.std())
q5["nonmetro_town"] = q5["rucc_2023"].between(4, 7).astype(int)
q5["rural"] = (q5["rucc_2023"] >= 8).astype(int)
q5["y2023"] = (q5["year"] == 2023).astype(int)
q5["y2024"] = (q5["year"] == 2024).astype(int)
q5["county_id"] = q5["fips"].astype("category").cat.codes

def fit_clustered(formula, data):
    return smf.ols(formula, data=data).fit(cov_type="cluster", cov_kwds={"groups": data["county_id"]})

def coef_table(fit):
    ci = fit.conf_int()
    return pd.DataFrame({"coef": fit.params, "p": fit.pvalues, "ci_low": ci[0], "ci_high": ci[1]}).round(3)

BARRIERS = "mean_km_to_site_z + nonmetro_town + rural + pct_senior_z + y2023 + y2024"
m_a = fit_clustered("divergence_z ~ " + BARRIERS, q5)
m_b = fit_clustered("divergence_z ~ " + BARRIERS + " + sites_per_10k_z", q5)
print(f"n = {int(m_a.nobs)} county-years | R2 barriers only: {m_a.rsquared:.3f} | with sites: {m_b.rsquared:.3f}")
pd.concat({"barriers only": coef_table(m_a), "with sites": coef_table(m_b)}, axis=1)

In [ ]:
import geopandas as gpd
tracts = gpd.read_file(cfg.EXTERNAL / "census" / "tract" / "tl_2020_27_tract" / "tl_2020_27_tract.shp")
counties_geo = tracts.dissolve(by="COUNTYFP")[["geometry"]].reset_index()
counties_geo["fips"] = "27" + counties_geo["COUNTYFP"]
geo = counties_geo.merge(div[["fips", "county_name", "need_rank", "div_mean", "need_index"]], on="fips", how="left")
stable7 = ["Mahnomen", "Clearwater", "Beltrami", "Wadena", "Ramsey", "Pine", "Nobles"]

fig, axes = plt.subplots(1, 2, figsize=(15, 8))
geo.plot(column="need_index", cmap="YlOrRd", legend=True, edgecolor="white", lw=0.4, ax=axes[0],
         legend_kwds={"label": "need index (higher = more need)", "shrink": 0.6})
geo[geo["county_name"].isin(stable7)].boundary.plot(ax=axes[0], color="black", lw=1.2)
axes[0].set_title("Need index, 2022-2024" + chr(10) + "Outlined: in the top ten under almost any weights", fontsize=10)

lim = geo["div_mean"].abs().max()
geo.plot(column="div_mean", cmap="RdBu_r", vmin=-lim, vmax=lim, legend=True, edgecolor="white", lw=0.4, ax=axes[1],
         legend_kwds={"label": "divergence: activity rank - need rank", "shrink": 0.6})
axes[1].set_title("Divergence" + chr(10) + "Red: need outranks recorded activity. Blue: the reverse", fontsize=10)
for _, r in geo.nlargest(6, "div_mean").iterrows():
    axes[1].annotate(r["county_name"], (r.geometry.centroid.x, r.geometry.centroid.y), ha="center", fontsize=7)
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

### Step 5b: barriers in neither need model
Three barriers that are inputs to neither Map the Meal Gap nor the index, each a reason a
household in need does not ask: households with limited English (ACS B16002), households
with no internet (B28002), non-citizens (B05001). None is correlated with poverty or with
SNAP (|r| under 0.4), so they are not the index wearing another name. Added as a second
block to the step 5 model.

| variable | mechanism | predicted sign |
|---|---|---|
| `pct_hh_limited_english` | cannot learn about or navigate a food shelf, so does not go | + |
| `pct_hh_no_internet` | no way to find out such help exists, so does not go | + |
| `pct_noncitizen` | less willing to engage with any program, so does not go. Correlated 0.93 with limited English, so the model will split one effect between the two; read them together | + |

Predictions written 2 October, before running. All three are barriers in the exact sense:
the mechanism ends in "does not go", which is what positive divergence means.

**Result (3 October):** with the three added, no-internet and distance are the only
barriers found to be significant, and more of either widens the gap between need and
recorded activity: no-internet +0.41 per SD (p = 0.005, clears the three-test bar of
0.017), distance +0.23 (p = 0.025, the pre-registered hypothesis now clearing 0.05).
R2 rises from 8.7% to 16.9% (adjusted 6.6% to 13.9%). Rural loses all of its effect: it
overlaps with no-internet (0.65), which takes what rural was carrying; which of the two
is the better description cannot be told with 87 counties. Limited English and
non-citizens split one effect between them (0.93 correlated) and come to nothing
together, so Nobles' divergence is not explained by its immigrant share here. Seniors
came out with the opposite sign to the prediction and an interval that includes zero:
nothing to build on. Step 5c then found no-internet is both a driver of need and a
barrier: higher index where internet is scarce, and lower activity for a given level of
need, the second narrowly (p = 0.049).

In [ ]:
for v in ["pct_hh_limited_english", "pct_hh_no_internet", "pct_noncitizen"]:
    q5[v] = df[v]
    q5[v + "_z"] = q5.groupby("year")[v].transform(lambda s: (s - s.mean()) / s.std())
NEW = "pct_hh_limited_english_z + pct_hh_no_internet_z + pct_noncitizen_z"
m_c = fit_clustered("divergence_z ~ " + BARRIERS + " + " + NEW, q5)
m_d = fit_clustered("divergence_z ~ " + NEW + " + y2023 + y2024", q5)
print(f"R2 | step 5 barriers: {m_a.rsquared:.3f} | plus the three new: {m_c.rsquared:.3f} | the three new alone: {m_d.rsquared:.3f}")
pd.concat({"step 5 barriers": coef_table(m_a), "with the three new": coef_table(m_c)}, axis=1)

### Step 5c: which side of the gap does no-internet explain?
The same three barriers explain 10.5% of the gap between the index and activity, but
1.0% of the gap between MMG and activity (Q1 appendix). Two readings: the index captures
need in low-internet counties that MMG misses and those households do not reach a food
shelf (the barrier explains the activity side); or the index is simply high where internet
is low, so no-internet "explains" divergence through the need side. The divergence is
`need_index - z_activity`, so regress each half on no-internet separately, with the other
half held fixed.

**Prediction, written before running (2 October):** the first reading. The index is not correlated with no-internet, so the barrier explains the activity side of the gap.

**Result (3 October):** both readings, about half each. The prediction failed on the need side: the index is correlated with no-internet (0.27 raw; +0.27 per SD with activity held fixed, p < 0.001), because no-internet, SNAP and no-vehicle households are all higher in the same poor rural counties. It held on the activity side: with the index held fixed, counties with more households without internet record less activity (-0.26 per SD, p = 0.049), which is the test of a barrier, passed narrowly. So households without internet mark counties where need is higher and, for a given level of need, recorded activity is lower. The 17% in step 5b is split between the two.

In [ ]:
q5["z_act"] = q5["need_index"] - q5["divergence_z"]
CTRL = "mean_km_to_site_z + nonmetro_town + rural + pct_senior_z + y2023 + y2024"
m_act = fit_clustered("z_act ~ need_index + pct_hh_no_internet_z + " + CTRL, q5)
m_need = fit_clustered("need_index ~ z_act + pct_hh_no_internet_z + " + CTRL, q5)
print("no-internet on the ACTIVITY side (index held fixed): coef {:.3f}, p {:.3f}".format(
      m_act.params["pct_hh_no_internet_z"], m_act.pvalues["pct_hh_no_internet_z"]))
print("no-internet on the NEED side (activity held fixed):  coef {:.3f}, p {:.3f}".format(
      m_need.params["pct_hh_no_internet_z"], m_need.pvalues["pct_hh_no_internet_z"]))
print("raw correlations with no-internet | index:", round(q5["need_index"].corr(q5["pct_hh_no_internet_z"]), 2),
      "| activity z:", round(q5["z_act"].corr(q5["pct_hh_no_internet_z"]), 2))
pd.concat({"activity side": coef_table(m_act), "need side": coef_table(m_need)}, axis=1)

## Step 6: the index against MMG, head to head
Q1 fitted log activity on log food-insecure persons. The index is not a count, so the
fair comparison is nested models on the same outcome, with county size held by log
population:

| model | answers |
|---|---|
| A: log activity ~ log population + MMG rate | how well MMG explains activity once size is out |
| B: log activity ~ log population + need index | the same for the index |
| C: both | does the index add anything MMG does not, and the reverse |

One fit per metric-year (nine), as in Q1; the table reports the mean R2 and the
coefficients and p-values in model C averaged over the nine fits. Then the residuals of
A and B give two rankings of "need not showing up as activity"; the counties that move
most between them are where the two measures disagree.

**Prediction, written before running (2 October):** both. The two complement each other:
each adds something the other does not.

**Result (3 October):** the prediction failed. County size explains 85.3% of recorded
activity; MMG adds 0.6 points, the index 0.4, both together 0.7, and neither is
significant with the other in the model. The two residual rankings correlate 0.99
because both are essentially the population residual; the largest mover shifts 12
places of 87 (Waseca, which the index sees as more under-recorded than MMG does; St.
Louis the reverse). The same finding from a third angle: recorded activity follows how
many people live in a county and almost nothing else, whichever way need is measured.

In [ ]:
rows, resid_a, resid_b = [], [], []
for met in cfg.THREE_METRICS:
    for yr in sorted(df["year"].unique()):
        g = df[(df["year"] == yr) & (df[met] > 0)].copy()
        g["log_act"], g["log_pop"] = np.log(g[met]), np.log(g["population"])
        fa = smf.ols("log_act ~ log_pop + fi_rate", data=g).fit(cov_type="HC3")
        fb = smf.ols("log_act ~ log_pop + need_index", data=g).fit(cov_type="HC3")
        fc = smf.ols("log_act ~ log_pop + fi_rate + need_index", data=g).fit(cov_type="HC3")
        rows.append({"metric": met, "year": yr, "R2 A (MMG)": fa.rsquared, "R2 B (index)": fb.rsquared,
                     "R2 C (both)": fc.rsquared, "R2 size only": smf.ols("log_act ~ log_pop", data=g).fit().rsquared,
                     "C: MMG coef": fc.params["fi_rate"], "C: MMG p": fc.pvalues["fi_rate"],
                     "C: index coef": fc.params["need_index"], "C: index p": fc.pvalues["need_index"]})
        resid_a.append(g[["county_name"]].assign(resid=fa.resid.values))
        resid_b.append(g[["county_name"]].assign(resid=fb.resid.values))
comp = pd.DataFrame(rows)
# rate on rate: activity per 1,000 (z, mean of metrics) on the two need measures together, no size term
rr = smf.ols("z_act ~ fi_rate + need_index", data=q5.assign(fi_rate=df["fi_rate"])).fit(cov_type="cluster", cov_kwds={"groups": q5["county_id"]})
print(f"rate on rate | R2 {rr.rsquared:.3f} | MMG rate p {rr.pvalues['fi_rate']:.3f} | index p {rr.pvalues['need_index']:.3f}")
print("mean over the nine metric-years:")
display(comp.drop(columns=["metric", "year"]).mean().round(3).to_frame("mean").T)
print("the index's p-value in model C, by metric-year:")
display(comp.pivot(index="metric", columns="year", values="C: index p").round(3))

ra = pd.concat(resid_a).groupby("county_name")["resid"].mean().rename("resid_mmg")
rb = pd.concat(resid_b).groupby("county_name")["resid"].mean().rename("resid_index")
two = pd.concat([ra, rb], axis=1)
two["rank_mmg"] = two["resid_mmg"].rank().astype(int)
two["rank_index"] = two["resid_index"].rank().astype(int)
two["moves"] = two["rank_index"] - two["rank_mmg"]
print("Spearman between the two residual rankings:", round(two["resid_mmg"].corr(two["resid_index"], method="spearman"), 3))
print("counties that look most under-recorded by the index but not by MMG (rank 1 = most under-recorded):")
display(two.sort_values("moves").head(8).round(2))
print("and the reverse:")
two.sort_values("moves").tail(6)[::-1].round(2)

### Three maps: where MMG puts need, where the index puts it, and where activity is
Same colour scale on all three: rank 1 (darkest) = most need, or most recorded activity
per 1,000 residents (mean of the three metrics, 2022-2024). Wilkin is hatched on the
activity map: no recorded activity. The counties named are the ones where MMG and the
index disagree most.

In [ ]:
mmg_rank = need.set_index("fips")["fi_rate"].rank(ascending=False).astype(int)
act_rank = div.set_index("fips")[[m + "_rank" for m in cfg.THREE_METRICS]].mean(axis=1).rank().astype(int)
geo3 = counties_geo.merge(need[["fips", "county_name", "need_rank"]], on="fips", how="left")
geo3["mmg_rank"] = geo3["fips"].map(mmg_rank)
geo3["activity_rank"] = geo3["fips"].map(act_rank)
geo3["gap"] = geo3["mmg_rank"] - geo3["need_rank"]
movers = geo3.reindex(geo3["gap"].abs().sort_values(ascending=False).index).head(8)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
panels = [("mmg_rank", "Need: Map the Meal Gap rate"), ("need_rank", "Need: the Q6 index"),
          ("activity_rank", "Recorded activity per 1,000 residents")]
for ax, (col, title) in zip(axes, panels):
    geo3.plot(column=col, cmap="YlOrRd_r", vmin=1, vmax=87, edgecolor="white", lw=0.4, ax=ax)
    if col == "activity_rank":
        geo3[geo3["county_name"] == "Wilkin"].plot(ax=ax, facecolor="none", edgecolor="black", hatch="///", lw=0.6)
    for k, (_, r) in enumerate(movers.iterrows()):
        ax.annotate(r["county_name"], (r.geometry.centroid.x, r.geometry.centroid.y),
                    xytext=(0, 6 if k % 2 else -6), textcoords="offset points", ha="center", fontsize=7)
    ax.set_title(title, fontsize=11)
    ax.set_axis_off()
sm = plt.cm.ScalarMappable(cmap="YlOrRd_r", norm=plt.Normalize(1, 87))
fig.colorbar(sm, ax=axes, shrink=0.6, pad=0.01, label="rank (1 = most)")
plt.show()

## Q6: answers
**1. The measure.** A household is in need if, at any point in the past year, it worried
about running out of food, regardless of whether it sought help. The county measure is an
equal-weight index of seven public inputs that are neither food-shelf demand nor inputs to
Map the Meal Gap: SNAP households, no-vehicle households, senior poverty, housing-cost
burden, adults without a diploma, low-income low-access share, and debt in collections.

**2. Against the three metrics.** The index behaves like MMG does against activity: the
counties with the highest need record less than expected, and per 1,000 residents the
relationship is almost flat (Spearman 0.07 to 0.14 for households, pounds and
individuals). Recorded demand does not rise with need, and that holds for all three
metrics alike. The index agrees with MMG's own rate broadly (Spearman 0.77) and disagrees
on five of the top ten counties.

**3. Where they diverge.** Wilkin, which has no recorded activity and ranks around the
70th percentile of need (predicted before the index was built); Nobles and Grant, high
need and low recorded activity, with Wadena, Benton, Chippewa and Todd behind them. In
the other direction Washington, Dodge, Cook and Houston record far more than their need.
Mahnomen is the exception that shows the measure can tell the two apart: first on need
and recording more than MMG predicts, a high-need county that is reached.

**4. What explains it.** Distance to a food shelf is the only barrier we think may lower
recorded activity relative to need (p = 0.075, the narrowest interval); the rural groups
are large and uncertain; seniors and site density show nothing. The barriers explain 9% of
the divergence, against 3% of Q1's residual.

**Stability.** The top seven (Mahnomen, Clearwater, Beltrami, Wadena, Ramsey, Pine,
Nobles) hold under every variant and in at least 71% of 500 random weightings; places 8 to
10 rotate among six counties. Wilkin's rank is weight-sensitive (2nd to 77th), which is
an argument for keeping equal weights.